# Creación de columnas analíticas

##### Georreferenciación, menores de edad, diversidad regional, correspondencia con rutas migratorias históricas, presencia de infraestructura de detención en municipios destino

### Georreferenciación

##### Esta sección fue realizada principalmente por @jdanielgoh

In [407]:
import geopandas as gpd
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
%pip install pyarrow

Note: you may need to restart the kernel to use updated packages.


In [408]:
from pathlib import Path

DATOS = Path("..") / "datos"
REFERENCIAS = DATOS / "referencias"

In [409]:
datos = pd.read_parquet(DATOS / "procesados" / "datos_limpios_concatenados.parquet")

In [410]:
catalogo_municipal = pd.read_csv(REFERENCIAS / "áreas_geoestadísticas_municipales.csv",encoding="latin-1",skiprows=2)

In [411]:
municipios = gpd.read_file(REFERENCIAS / "municipios" / "00mun.shp")

municipios = municipios.to_crs(epsg=4326)

In [412]:
catalogo_municipal["mun"] = catalogo_municipal.apply(lambda x : x["Nombre del Municipio"] + ", " + x["Nombre del Estado"] ,axis=1)
catalogo_municipal["cve_mun"] = catalogo_municipal.apply(lambda x : str(x["Clave Geoestadística Estatal"]).zfill(2) + str(x["Clave Geoestadística Municipal"]).zfill(3) ,axis=1)
diccionario_nom_cve = catalogo_municipal.set_index("mun").cve_mun.to_dict()

diccionario_cve_geom = municipios.set_index("CVEGEO").geometry.to_dict()

In [413]:
diccionario_nombres_oficiales = {'Michoacán':'Michoacán de Ocampo',
       "Veracruz":'Veracruz de Ignacio de la Llave',"Estado de México":"México"}

def limpiadoraLugares(txt):
  txt = txt.replace("El Ceibo","Tenosique")
  txt = txt.replace("Ciudad Victoria","Victoria")

  if(txt=="Ciudad del Carmen, Campeche"):
    txt = "Carmen, Campeche"
  if(txt=="Ciudad Juárez, Chihuahua"):
    txt = "Juárez, Chihuahua"
  if(txt=="Ciudad Cuauhtémoc, Chihuahua"):
    txt = "Cuauhtémoc, Chihuahua"
  if(txt=="Acapulco, Guerrero"):
    txt = "Acapulco de Juárez, Guerrero"

  if(txt=="Tlalnepantla, Estado de México"):
    txt = "Tlalnepantla de Baz, Estado de México"

  if(txt=="Talismán, Chiapas"):
    txt = "Tuxtla Chico, Chiapas"
  if(txt=="Pachuca de Soto, Hidalgo"):
    txt = "Pachuca de Soto, Hidalgo"
  if(txt=="El Carmen, Chiapas"):
    txt="Pijijiapan, Chiapas"
  if(txt=="Plan de Guadalupe, Nuevo León"):
    txt="Guadalupe, Nuevo León"
  if(txt=="Echegaray, Chiapas"):
    txt="Pijijiapan, Chiapas"
  if(txt=="Piedras Negras, Coahuila"):
    txt="Piedras Negras, Coahuila de Zaragoza"
  if(txt=="Saltillo, Coahuila"):
    txt="Saltillo, Coahuila de Zaragoza"
  if(txt=="Escuinapa de Hidalgo, Sinaloa"):
    txt="Escuinapa, Sinaloa"
  if(txt=='Guadalupe, Nuevo León  '):
    txt='Guadalupe, Nuevo León'
      
  txt_sep = txt.split(", ")
  if(txt_sep[-1] in diccionario_nombres_oficiales.keys()):
    txt_sep[-1]  = diccionario_nombres_oficiales[txt_sep[-1]]
  return ", ".join(txt_sep)

datos["origen_limpio_mun"] = datos.lugar_origen.apply(limpiadoraLugares).apply(lambda x : x.split(", ")[0])
datos["origen_limpio_ent"] = datos.lugar_origen.apply(limpiadoraLugares).apply(lambda x : x.split(", ")[-1])
datos["destino_limpio_mun"] = datos.lugar_destino.apply(limpiadoraLugares).apply(lambda x : x.split(", ")[0])
datos["destino_limpio_ent"] = datos.lugar_destino.apply(limpiadoraLugares).apply(lambda x : x.split(", ")[-1])

datos["destino_cve"] = datos.lugar_destino.apply(limpiadoraLugares).map(diccionario_nom_cve)

datos["origen_cve"] = datos.lugar_origen.apply(limpiadoraLugares).map(diccionario_nom_cve)

In [414]:
datos[datos.origen_cve.isna()].lugar_origen.value_counts()

Series([], Name: count, dtype: int64)

In [415]:
datos[datos.destino_cve.isna()].lugar_destino.value_counts()

lugar_destino
Honduras          1039
Guatemala          991
El Salvador         21
Deportación          2
Estados Unidos       2
Belice               1
Nicaragua            1
Name: count, dtype: int64

In [416]:
datos["geom_destino"] = datos.destino_cve.map(diccionario_cve_geom).apply(lambda x : x.centroid if type(x)!=float else np.nan)
datos["geom_origen"] = datos.origen_cve.map(diccionario_cve_geom).apply(lambda x : x.centroid if type(x)!=float else np.nan)

In [417]:
datos["lon_destino"] = datos["geom_destino"].apply(lambda x : x.x if type(x)!=float else np.nan )
datos["lat_destino"] = datos["geom_destino"].apply(lambda x : x.y if type(x)!=float else np.nan )

datos["lon_origen"] = datos["geom_origen"].apply(lambda x : x.x if type(x)!=float else np.nan )
datos["lat_origen"] = datos["geom_origen"].apply(lambda x : x.y if type(x)!=float else np.nan )

In [418]:
datos[["lon_origen","lat_origen"]].groupby(["lon_origen","lat_origen"]).size().reset_index().rename({0:"cant"},axis=1)

,lon_origen,lat_origen,cant
0,-116.847384,32.412116,200
1,-111.473211,29.024027,507
2,-109.162235,25.935276,4
3,-108.549129,25.518076,3
4,-108.450224,30.849353,239
...,...,...,...
72,-90.474350,18.550598,23
73,-90.266050,19.616416,6
74,-89.639925,20.980519,22
75,-88.466935,18.508613,54


In [419]:
datos[["lon_origen","lat_origen","lon_destino","lat_destino"]].groupby(["lon_origen","lat_origen","lon_destino","lat_destino"]).size().reset_index().rename({0:"cant"},axis=1)

,lon_origen,lat_origen,lon_destino,lat_destino,cant
0,-116.847384,32.412116,-116.847384,32.412116,1
1,-116.847384,32.412116,-111.473211,29.024027,8
2,-116.847384,32.412116,-106.306029,23.468303,7
3,-116.847384,32.412116,-102.678068,22.735737,7
4,-116.847384,32.412116,-100.975944,22.289290,13
...,...,...,...,...,...
432,-88.466935,18.508613,-88.466935,18.508613,1
433,-86.940930,21.078221,-93.137285,16.739447,1
434,-86.940930,21.078221,-92.976049,16.606635,1
435,-86.940930,21.078221,-92.860211,18.009104,2


In [420]:
print(datos.columns)

Index(['id_fila_og', 'year', 'oficina_regional', 'lugar_origen',
       'lugar_destino', 'total_migrantes_traslado', 'nacionalidad', 'hombres',
       'mujeres', 'niñas', 'niños', 'regla', 'id_viaje', 'origen_limpio_mun',
       'origen_limpio_ent', 'destino_limpio_mun', 'destino_limpio_ent',
       'destino_cve', 'origen_cve', 'geom_destino', 'geom_origen',
       'lon_destino', 'lat_destino', 'lon_origen', 'lat_origen'],
      dtype='object')


# Asignación de sentidos

### Veamos las combinaciones de origenes y destinos y les asignamos un sentido manualmente en el diccionario de sentidos; luego, usamos el diccionario para asignar valores a la tabla maestra

In [421]:
combinaciones = datos[['origen_cve', 'destino_cve', 'lugar_origen', 'lugar_destino']].drop_duplicates(
    subset=['origen_cve', 'destino_cve']
)

combinaciones['sentido'] = None  

combinaciones = combinaciones.reset_index(drop=True)
combinaciones.index = combinaciones.index + 1  

with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'display.max_colwidth', None):
    display(combinaciones)

,origen_cve,destino_cve,lugar_origen,lugar_destino,sentido
1,08035,08019,"Janos, Chihuahua","Chihuahua, Chihuahua",None
2,08035,09007,"Janos, Chihuahua","Iztapalapa, Ciudad de México",None
3,13048,09007,"Pachuca de Soto, Hidalgo","Iztapalapa, Ciudad de México",None
4,14039,30003,"Guadalajara, Jalisco","Acayucan, Veracruz",None
5,15106,09007,"Toluca, México","Iztapalapa, Ciudad de México",None
6,20327,07089,"San Pedro Tapanatepec, Oaxaca","Tapachula, Chiapas",None
7,20043,NaN,"Heroica Ciudad de Juchitán de Zaragoza, Oaxaca",Guatemala,None
8,23004,23004,"Othón P. Blanco, Quintana Roo","Othón P. Blanco, Quintana Roo",None
9,24028,09007,"San Luis Potosí, San Luis Potosí","Iztapalapa, Ciudad de México",None
10,24028,07089,"San Luis Potosí, San Luis Potosí","Tapachula, Chiapas",None


In [422]:
# Asegurar que CVEGEO sea un string con 5 caracteres
combinaciones['origen_cve'] = combinaciones['origen_cve'].astype(str).str.split('.').str[0].str.zfill(5)
combinaciones['destino_cve'] = combinaciones['destino_cve'].astype(str).str.split('.').str[0].str.zfill(5)

In [423]:
# Aplicamos el diccionario previamente asignado de manera manual con referencia al mapa de rutas generado en QGIS
dic = pd.read_parquet(REFERENCIAS / "diccionario_sentidos.parquet")

dict_sentido = (
    dic
    .drop_duplicates(subset=['origen_cve', 'destino_cve'])
    .set_index(['origen_cve', 'destino_cve'])['sentido']
    .to_dict()
)

combinaciones['sentido'] = combinaciones.apply(
    lambda x: "dep" if pd.isna(x['destino_cve']) or x['destino_cve'] == "00nan" else dict_sentido.get((x['origen_cve'], x['destino_cve'])),
    axis=1
)

In [424]:
# Verificar
combinaciones[combinaciones['sentido'].isna()]

,origen_cve,destino_cve,lugar_origen,lugar_destino,sentido


In [425]:
# Aplicar a datos 
datos['origen_cve'] = datos['origen_cve'].astype('string').str.split('.').str[0].str.zfill(5)
datos['destino_cve'] = datos['destino_cve'].astype('string').str.split('.').str[0].str.zfill(5)

datos['sentido'] = datos.apply(
    lambda x: "dep" if pd.isna(x['destino_cve']) else dict_sentido.get((x['origen_cve'], x['destino_cve'])),
    axis=1
)

In [426]:
# Verificar
datos[datos['sentido'].isna()]

,id_fila_og,year,oficina_regional,lugar_origen,lugar_destino,total_migrantes_traslado,nacionalidad,hombres,mujeres,niñas,...,destino_limpio_ent,destino_cve,origen_cve,geom_destino,geom_origen,lon_destino,lat_destino,lon_origen,lat_origen,sentido


In [427]:
# Chequeo manual
datos[['sentido', 'lugar_origen', 'lugar_destino']].sample(30)

,sentido,lugar_origen,lugar_destino
3647,dep,"Acayucan, Veracruz",Honduras
10127,dep,"Centro, Tabasco",Honduras
9320,ret,"Tijuana, Baja California","Iztapalapa, Ciudad de México"
288,ret,"Tijuana, Baja California","Iztapalapa, Ciudad de México"
4249,ret,"Janos, Chihuahua","Acayucan, Veracruz"
1215,lat,"Tenosique, Tabasco","Tapachula, Chiapas"
440,ret,"Piedras Negras, Coahuila","Iztapalapa, Ciudad de México"
6064,dep,"Acayucan, Veracruz",Honduras
1908,ret,"Reynosa, Tamaulipas","Tampico, Tamaulipas"
10855,ret,"Hermosillo, Sonora","Centro, Tabasco"


# Creación de demás columnas analíticas

### Menores

In [428]:
# Creamos las columnas menores_num y menores_bin 
datos['menores_num'] = datos['niñas'] + datos['niños']

# Checar trabajo, menores_num debería = niñas + niños 
print(datos[['niñas', 'niños', 'menores_num']].head(15))

    niñas  niños  menores_num
0       0      0            0
1       0      0            0
2       9      3           12
3       6      4           10
4       2      4            6
5       0      0            0
6       7     25           32
7       4      0            4
8       5      1            6
9       3      0            3
10      1      2            3
11      3     16           19
12      1     15           16
13      4      7           11
14     17     34           51


In [429]:
# Crear menores_bin
datos['menores_bin'] = np.where(datos['menores_num'] > 0, 1, 0)

# Checar trabajo, menores_bin debería = 1 cuando hubieron menores en un dado traslado, y 0 cuando no
print(datos[['menores_num', 'menores_bin']].head(10))

   menores_num  menores_bin
0            0            0
1            0            0
2           12            1
3           10            1
4            6            1
5            0            0
6           32            1
7            4            1
8            6            1
9            3            1


### Regiones

In [430]:
# Creamos columnas binarias para mostrar la presencia / ausencia de migrantes de diferentes regiones en cada traslado
!pip install rapidfuzz
from rapidfuzz import process

In [431]:
# Creamos un diccionario para asignar los países a las regiones
nationality_to_region = {
    # Norteamérica
    'mexicana': 'norteamerica',
    'mexico': 'norteamerica',
    'estadounidense': 'norteamerica',
    'estados unidos': 'norteamerica',
    'canadiense': 'norteamerica',
    'canada': 'norteamerica',

    # Centroamérica
    'guatemalteca': 'centroamerica',
    'guatemala': 'centroamerica',
    'hondurena': 'centroamerica',
    'hondureña': 'centroamerica',
    'honduras': 'centroamerica',
    'salvadorena': 'centroamerica',
    'salvadoreña': 'centroamerica',
    'el salvador': 'centroamerica',
    'salvador': 'centroamerica',
    'nicaraguense': 'centroamerica',
    'nicaragua': 'centroamerica',
    'costarricense': 'centroamerica',
    'costa rica': 'centroamerica',
    'panamena': 'centroamerica',
    'panameña': 'centroamerica',
    'panama': 'centroamerica',
    'belice': 'centroamerica',
    'beliceña': 'centroamerica',

    # Sudamérica 
    'venezolana': 'sudamerica',
    'venezuela': 'sudamerica',
    'colombiana': 'sudamerica',
    'colombia': 'sudamerica',
    'ecuatoriana': 'sudamerica',
    'ecuador': 'sudamerica',
    'peruana': 'sudamerica',
    'peru': 'sudamerica',
    'boliviana': 'sudamerica',
    'bolivia': 'sudamerica',
    'chilena': 'sudamerica',
    'chile': 'sudamerica',
    'argentina': 'sudamerica',
    'uruguaya': 'sudamerica',
    'uruguay': 'sudamerica',
    'paraguaya': 'sudamerica',
    'paraguay': 'sudamerica',
    'brasilena': 'sudamerica',
    'brasil': 'sudamerica',
    'guyana': 'sudamerica',
    'guyanesa': 'sudamerica',
    'suriname': 'sudamerica',

    # Caribe
    'cubana': 'caribe',
    'cuba': 'caribe',
    'dominicana': 'caribe',
    'republica dominicana': 'caribe',
    'rep dominicana': 'caribe',
    'haitiana': 'caribe',
    'haiti': 'caribe',
    'puertorriquena': 'caribe',
    'puerto rico': 'caribe',
    'trinidad y tobago': 'caribe',
    'trinidad': 'caribe',
    'antillano': 'caribe',
    'antillas': 'caribe',
    'curucao': 'caribe',
    'islas virgenes': 'caribe',
    'jamaica': 'caribe',
    'jamaiquino': 'caribe',
    'bahamas': 'caribe',
    'bahameña': 'caribe',

    # Extracontinentales
    'francesa': 'extracontinentales',
    'francia': 'extracontinentales',
    'alemana': 'extracontinentales',
    'alemania': 'extracontinentales',
    'italiana': 'extracontinentales',
    'italia': 'extracontinentales',
    'japonesa': 'extracontinentales',
    'japon': 'extracontinentales',
    'china': 'extracontinentales',
    'coreana': 'extracontinentales',
    'corea': 'extracontinentales',
    'nigeriana': 'extracontinentales',
    'nigeria': 'extracontinentales',
    'india': 'extracontinentales',
    'hindu': 'extracontinentales',
    'angola': 'extracontinentales',
    'angoleña': 'extracontinentales',
    'camerun': 'extracontinentales',
    'rusia': 'extracontinentales', 
    'rusa': 'extracontinentales',
    'ucraniana': 'extracontinentales',
    'ucrania': 'extracontinentales',
    'polaca': 'extracontinentales',
    'polonia': 'extracontinentales',
    'bielorrusia': 'extracontinentales',
    'bielorusa': 'extracontinentales',
    'libanesa': 'extracontinentales',
    'libano': 'extracontinentales',
    'siria': 'extracontinentales',
    'palestina': 'extracontinentales',
    'libia': 'extracontinentales',
    'mauritania': 'extracontinentales',
    'congoleña': 'extracontinentales',
    'congo': 'extracontinentales',
    'somalia': 'extracontinentales',
    'somali': 'extracontinentales',
    'malaysia': 'extracontinentales',
    'malay': 'extracontinentales',
    'bengali': 'extracontinentales',
    'bengal': 'extracontinentales', 
    'armenia': 'extracontinentales',
    'mongol': 'extracontinentales',
    'mongolia': 'extracontinentales',
    'turca': 'extracontinentales',
    'turquia': 'extracontinentales',
    'yemeni': 'extracontinentales',
    'yemen': 'extracontinentales',
    'uzbekistan': 'extracontinentales',
    'nepali': 'extracontinentales',
    'senegal': 'extracontinentales'
}

known_nationalities = list(nationality_to_region.keys())

In [432]:
import re
from rapidfuzz import process

stopwords = {'y', 'e', 'o', 'u'}  # ignorar conectores

def fuzzy_flag_regions(nationality_str, threshold=80):
    # empezar desde 0
    region_flags = {
        'norteamerica': 0,
        'centroamerica': 0,
        'sudamerica': 0,
        'caribe': 0,
        'extracontinentales': 0,
        'nan_region': 0
    }

    # sobre NaN
    if pd.isna(nationality_str):
        return pd.Series(region_flags)
    
    nationality_str = nationality_str.strip().lower()

    # asigna filas con "varios" a NaN
    if nationality_str == 'varios':
        region_flags['nan_region'] = 1
        return pd.Series(region_flags)

    # proceso
    tokens = re.findall(r'\b[a-zA-Záéíóúñ]+\b', nationality_str)

    for token in tokens:
        if len(token) < 3 or token in stopwords:
            continue
        
        match, score, _ = process.extractOne(token, known_nationalities)
        if score >= threshold:
            region = nationality_to_region[match]
            region_flags[region] = 1
        else:
            region_flags['extracontinentales'] = 1

    return pd.Series(region_flags)

In [433]:
# Agregar regiones
region_flags_datos = datos['nacionalidad'].apply(fuzzy_flag_regions)

datos_menores_regiones = pd.concat([datos, region_flags_datos], axis=1)

In [434]:
# Limpiar ortografía
datos_menores_regiones.columns = datos_menores_regiones.columns.str.strip()

# Verificar que aparezcan las columnas que son
print(datos_menores_regiones.columns.tolist())

['id_fila_og', 'year', 'oficina_regional', 'lugar_origen', 'lugar_destino', 'total_migrantes_traslado', 'nacionalidad', 'hombres', 'mujeres', 'niñas', 'niños', 'regla', 'id_viaje', 'origen_limpio_mun', 'origen_limpio_ent', 'destino_limpio_mun', 'destino_limpio_ent', 'destino_cve', 'origen_cve', 'geom_destino', 'geom_origen', 'lon_destino', 'lat_destino', 'lon_origen', 'lat_origen', 'sentido', 'menores_num', 'menores_bin', 'norteamerica', 'centroamerica', 'sudamerica', 'caribe', 'extracontinentales', 'nan_region']


In [435]:
# ¿Cuántas respuestas NaN hay?
datos_menores_regiones['nan_region'].sum()

np.int64(0)

In [436]:
# Chequeo manual de 20 filas random
pd.set_option('display.max_colwidth', None)

columns_to_show = ['nacionalidad', 'norteamerica', 'centroamerica', 'sudamerica', 'caribe', 'extracontinentales', 'nan_region']
display(datos_menores_regiones[columns_to_show].sample(20))

,nacionalidad,norteamerica,centroamerica,sudamerica,caribe,extracontinentales,nan_region
997,26 hondureños 4 guatemaltecos 02 salvadoreños,0,1,0,0,0,0
7065,"03 colombia, 14 guatemala, 01 ecuador, 02 salvadoreña",0,1,1,0,0,0
4473,40 honduras 01 guatemala,0,1,0,0,0,0
374,hondureña y dos de españa,1,1,0,0,1,0
6840,"10 haiti, 31 venezuela",0,0,1,1,0,0
2332,nicaraguense,0,1,0,0,0,0
1430,"62 honduras, 08 guatemala, 03 salvador, 02 nicaragua",0,1,0,0,0,0
5284,guatemalteca,0,1,0,0,0,0
4050,hondureña,0,1,0,0,0,0
4019,guatemalteca,0,1,0,0,0,0


### Diversidad regional

In [437]:
# Crear columna de diversidad - la suma de las columnas binarias de regiones - muestra de cuántas regiones venían en cada traslado

cols = ['norteamerica', 'centroamerica', 'sudamerica', 'caribe', 'nan_region', 'extracontinentales']
datos_menores_regiones['diversity'] = datos_menores_regiones[cols].sum(axis=1)

In [438]:
# Agregar columna de categorias de diversidad - sólo una región o múltiples 
datos_menores_regiones['diversity_cat'] = datos_menores_regiones['diversity'].apply(
    lambda x: 'Una región' if x == 1 else 'Dos o más regiones'
)

In [439]:
# Muestra random para chequeo manual
datos_menores_regiones[['norteamerica', 'centroamerica', 'sudamerica', 'caribe', 'nan_region', 'extracontinentales', 'diversity', 'diversity_cat']].sample(20)

,norteamerica,centroamerica,sudamerica,caribe,nan_region,extracontinentales,diversity,diversity_cat
5807,0,1,0,0,0,0,1,Una región
3308,0,1,0,0,0,0,1,Una región
2131,0,1,0,0,0,0,1,Una región
5388,0,1,0,0,0,0,1,Una región
5611,0,1,1,0,0,0,2,Dos o más regiones
8947,0,1,1,1,0,0,3,Dos o más regiones
1991,0,1,0,0,0,0,1,Una región
3204,0,1,0,0,0,0,1,Una región
492,0,1,0,0,0,0,1,Una región
4512,0,1,0,0,0,0,1,Una región


In [440]:
datos_menores_regiones.to_csv(DATOS / "procesados" / "datos_analizados_condep.csv", index=False)

### Correspondencia

#### Esta tabla que importamos se hizo en base del mapa de rutas en QGIS. Hay 9 rutas documentadas (4 de sur a centro, 5 de centro al norte) y la tabla les enlista con el nombre de cada ruta y la lista extraída de los municipios y entidades a que atraviesa la línea + sus buffers de 5km 

#### El proceso para el análisis de rutas se describe en el diccionario de datos, vinculado en la carpeta de documentos

In [441]:
# Primero, verificamos que cualquier NaN en columna "destino" se debe a que es una deportación
es_nulo = datos_menores_regiones['destino_cve'].isna()

es_texto_nan = (
    datos_menores_regiones['destino_cve']
    .astype(str).str.lower()
    .str.contains(r'nan|none|<na>', regex=True)
)

sin_destino = es_nulo | es_texto_nan

datos_menores_regiones.loc[sin_destino, 'lugar_destino'].unique()

array(['Guatemala', 'Honduras', 'El Salvador', 'Deportación', 'Belice',
       'Estados Unidos', 'Nicaragua'], dtype=object)

In [442]:
# Recortar tabla de datos para graficar para sólo incluir los traslados domésticos y excluir las deportaciones
datos_mx = datos_menores_regiones[~sin_destino].copy()

# Asegurar que CVEGEOs son en el formato correcto para ser interpretados
datos_mx['origen_cve'] = datos_mx['origen_cve'].astype(float).astype(int).astype(str).str.zfill(5)
datos_mx['destino_cve'] = datos_mx['destino_cve'].astype(float).astype(int).astype(str).str.zfill(5)

In [443]:
rutas = pd.read_csv(REFERENCIAS / "rutas_buff/rutas_buff_mun_master.csv")

In [444]:
# Crear diccionario para poder identificar si una localidad está o no en alguna ruta y cuál 
cvegeo_to_routes = {}

for idx, row in rutas.iterrows():
    cvegeo_list = [code.strip() for code in row['CVEGEO_lista'].split(',')]
    
    for cvegeo in cvegeo_list:
        if cvegeo not in cvegeo_to_routes:
            cvegeo_to_routes[cvegeo] = []
        cvegeo_to_routes[cvegeo].append(row['ruta'])

datos_mx['rutas_posibles_o'] = datos_mx['origen_cve'].map(
    lambda x: ', '.join(cvegeo_to_routes.get(x, []))
)

datos_mx['rutas_posibles_d'] = datos_mx['destino_cve'].map(
    lambda x: ', '.join(cvegeo_to_routes.get(x, []))
)

print(datos_mx[['lugar_origen', 'lugar_destino', 'rutas_posibles_o', 'rutas_posibles_d']].sample(20))

                                    lugar_origen  \
9304                    Tijuana, Baja California   
5630                         Reynosa, Tamaulipas   
980                 Cuauhtémoc, Ciudad de México   
179                     Nuevo Laredo, Tamaulipas   
7510                    Piedras Negras, Coahuila   
10773                         Hermosillo, Sonora   
31                 San Pedro Tapanatepec, Oaxaca   
928                     Tijuana, Baja California   
10578                      Guadalupe, Nuevo León   
9697               San Pedro Tapanatepec, Oaxaca   
7959                              Puebla, Puebla   
10381                       Zacatecas, Zacatecas   
4443              Monclova, Coahuila de Zaragoza   
7655                    Piedras Negras, Coahuila   
8803   Apetatitlán de Antonio Carvajal, Tlaxcala   
525                            Mazatlán, Sinaloa   
8555                         Tampico, Tamaulipas   
9669                   Álvaro Obregón, Michoacán   
2756        

In [445]:
# Si no hubo rutas posibles, poner "ninguna"
datos_mx['rutas_posibles_o'] = datos_mx['rutas_posibles_o'].replace('', 'ninguna')
datos_mx['rutas_posibles_d'] = datos_mx['rutas_posibles_d'].replace('', 'ninguna')

In [446]:
# Agrupar rutas en nortes y sures
rutas_norte = {'baja', 'sonora', 'chihuahua', 'tejas', 'golfo'}
rutas_sur = {'pacifico', 'chiapas', 'tabasco', 'caribe'}

def get_route_set(route_string):
    """Convert 'ruta1, ruta2' to {'ruta1', 'ruta2'}"""
    if route_string == 'ninguna' or route_string == '' or pd.isna(route_string):
        return set()
    return {r.strip().lower() for r in route_string.split(',') if r.strip()}

# Definir correspondencia entre rutas de origen y rutas de destino
def clasificar_correspondencia(row):
    rutas_o = get_route_set(row['rutas_posibles_o'])
    rutas_d = get_route_set(row['rutas_posibles_d'])
    
   # Casos con un valor "ninguna"
    if not rutas_o and not rutas_d:
        return 'ambos_fuera'
    if not rutas_o:
        return 'o_fuera'
    if not rutas_d:
        return 'd_fuera'

    # Casos de coincidencia
    if rutas_o & rutas_d: 
        return 'ruta_cont_1'
    
    # Checar norte y sur
    o_is_norte = rutas_o.issubset(rutas_norte)
    o_is_sur = rutas_o.issubset(rutas_sur)
    d_is_norte = rutas_d.issubset(rutas_norte)
    d_is_sur = rutas_d.issubset(rutas_sur)
    
    # Uno y uno; casos particulares de rutas definidas como principales
    if (o_is_norte and d_is_sur) or (o_is_sur and d_is_norte):
        # Checar si está pacifico O (tabasco y golfo)
        todas_rutas = rutas_o | rutas_d
        if 'pacifico' in todas_rutas or ('tabasco' in todas_rutas and 'golfo' in todas_rutas):
            return 'ruta_cont_1'
        else:
            return 'ruta_cont_2'
    
    # Dos nortes o dos sures
    if (o_is_norte and d_is_norte) or (o_is_sur and d_is_sur):
        return 'rutas_diferentes'
    
    # Fallas
    return 'ERROR'

# Crear columna
datos_mx['correspondencia'] = datos_mx.apply(clasificar_correspondencia, axis=1)

# Checar resultados
print(datos_mx[['rutas_posibles_o', 'rutas_posibles_d', 'correspondencia']].head(20))

# Checar errores
print("\nCorrespondencia:")
print(datos_mx['correspondencia'].value_counts())

           rutas_posibles_o                rutas_posibles_d correspondencia
0                   ninguna                       chihuahua         o_fuera
1                   ninguna                       chihuahua         o_fuera
2                   ninguna  baja, chihuahua, sonora, tejas         o_fuera
3                   ninguna  baja, chihuahua, sonora, tejas         o_fuera
4                   ninguna                       chihuahua         o_fuera
5                   ninguna                       chihuahua         o_fuera
6                   ninguna  baja, chihuahua, sonora, tejas         o_fuera
7              baja, sonora                 caribe, tabasco     ruta_cont_2
8              baja, sonora                 caribe, tabasco     ruta_cont_2
9              baja, sonora                 caribe, tabasco     ruta_cont_2
10  baja, chihuahua, sonora  baja, chihuahua, sonora, tejas     ruta_cont_1
11  baja, chihuahua, sonora  baja, chihuahua, sonora, tejas     ruta_cont_1
12  baja, ch

In [447]:
# Revisar nombres de municipio cuando hubo errores
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

errores = datos_mx[datos_mx['correspondencia'] == 'ERROR'][['lugar_origen', 'lugar_destino']]
print(errores)

print(f"\nErrores: {len(errores)}")

pd.reset_option('display.max_rows')
pd.reset_option('display.max_columns')
pd.reset_option('display.width')
pd.reset_option('display.max_colwidth')

            lugar_origen       lugar_destino
2825      Puebla, Puebla     Centro, Tabasco
2830      Puebla, Puebla     Centro, Tabasco
2831      Puebla, Puebla     Centro, Tabasco
2832      Puebla, Puebla     Centro, Tabasco
2833      Puebla, Puebla     Centro, Tabasco
2836      Puebla, Puebla     Centro, Tabasco
2838      Puebla, Puebla     Centro, Tabasco
2840      Puebla, Puebla     Centro, Tabasco
2845      Puebla, Puebla     Centro, Tabasco
2848      Puebla, Puebla     Centro, Tabasco
2849      Puebla, Puebla     Centro, Tabasco
2850      Puebla, Puebla  Acayucan, Veracruz
2852      Puebla, Puebla     Centro, Tabasco
2853      Puebla, Puebla  Acayucan, Veracruz
2855      Puebla, Puebla  Acayucan, Veracruz
2858      Puebla, Puebla  Acayucan, Veracruz
2860      Puebla, Puebla  Acayucan, Veracruz
2867      Puebla, Puebla     Centro, Tabasco
2868      Puebla, Puebla     Centro, Tabasco
2878      Puebla, Puebla     Centro, Tabasco
2887      Puebla, Puebla     Centro, Tabasco
2969      

#### Muchos de los errores eran porque hay un punto conector entre las redes de rutas del norte y del sur: Puebla. Hay que clasificarlo como parte de una red o la otra. La sacaremos de las rutas del sur

In [448]:
rutas_sur_names = ['pacifico', 'chiapas', 'tabasco', 'caribe']

for ruta_name in rutas_sur_names:
    # Actualiarzar CVEGEO_lista - sacar 21114
    rutas.loc[rutas['ruta'] == ruta_name, 'CVEGEO_lista'] = rutas.loc[rutas['ruta'] == ruta_name, 'CVEGEO_lista'].str.replace(r',?\s*21114\s*,?', '', regex=True).str.strip(', ')
    
    # Actualizar municipio_nombre - sacar Puebla
    rutas.loc[rutas['ruta'] == ruta_name, 'municipios_nombre'] = rutas.loc[rutas['ruta'] == ruta_name, 'municipios_nombre'].str.replace(r',?\s*Puebla\s*,?', '', regex=True).str.strip(', ')

# Verificar
print(rutas[rutas['ruta'].isin(rutas_sur_names)][['ruta', 'CVEGEO_lista', 'municipios_nombre']])

       ruta                                       CVEGEO_lista  \
1    caribe  04003, 04007, 04004, 04002, 04006, 07074, 2027...   
3   chiapas  07019, 07108, 07106, 07104, 07075, 07086, 0708...   
5  pacifico  07103, 07102, 07057, 07054, 07051, 07070, 0705...   
7   tabasco  07074, 07050, 07065, 07077, 07016, 20278, 2018...   

                                   municipios_nombre  
1  Carmen, Palizada, Champotón, Campeche, Hopelch...  
3  Comitán de Domínguez, Villaflores, Venustiano ...  
5  Tuzantán, Tuxtla Chico, Motozintla, Mazatán, M...  
7  Reforma, La Libertad, Palenque, Salto de Agua,...  


In [449]:
# Rehacer diccionario de rutas
cvegeo_to_routes = {}

for idx, row in rutas.iterrows():
    cvegeo_list = [code.strip() for code in row['CVEGEO_lista'].split(',')]
    for cvegeo in cvegeo_list:
        if cvegeo not in cvegeo_to_routes:
            cvegeo_to_routes[cvegeo] = []
        cvegeo_to_routes[cvegeo].append(row['ruta'])

datos_mx['rutas_posibles_o'] = datos_mx['origen_cve'].map(
    lambda x: ', '.join(cvegeo_to_routes.get(x, [])) or 'ninguna'
)

datos_mx['rutas_posibles_d'] = datos_mx['destino_cve'].map(
    lambda x: ', '.join(cvegeo_to_routes.get(x, [])) or 'ninguna'
)

# Recalcular
datos_mx['correspondencia'] = datos_mx.apply(clasificar_correspondencia, axis=1)

# Checar errores
print(datos_mx['correspondencia'].value_counts())

correspondencia
ruta_cont_1         3917
ruta_cont_2         2149
rutas_diferentes    1129
o_fuera             1093
d_fuera              526
ambos_fuera           15
Name: count, dtype: int64


In [450]:
# Checar lugar_origen donde correspondencia es o_fuera
origenes_fuera = datos_mx[datos_mx['correspondencia'] == 'o_fuera']['lugar_origen'].unique()
print("Orígenes fuera de ruta:")
print(list(origenes_fuera))
print(f"\nTotal: {len(origenes_fuera)}")

print("\n" + "="*50 + "\n")

# Checar lugar_destino donde correspondencia es d_fuera
destinos_fuera = datos_mx[datos_mx['correspondencia'] == 'd_fuera']['lugar_destino'].unique()
print("Destinos fuera de ruta:")
print(list(destinos_fuera))
print(f"\nTotal: {len(destinos_fuera)}")

Orígenes fuera de ruta:
['Janos, Chihuahua', 'Pachuca de Soto, Hidalgo', 'Escárcega, Campeche', 'Tuxtla Gutiérrez, Chiapas', 'Santa María Huatulco, Oaxaca', 'Salina Cruz, Oaxaca', 'Othón P. Blanco, Quintana Roo', 'Apetatitlán de Antonio Carvajal, Tlaxcala', 'San Cristóbal de las Casas, Chiapas', 'Acapulco de Juárez, Guerrero', 'Mérida, Yucatán', 'Guanajuato, Guanajuato', 'Apaxco, México', 'Tlaxcala, Tlaxcala']

Total: 14


Destinos fuera de ruta:
['Othón P. Blanco, Quintana Roo', 'Pachuca de Soto, Hidalgo', 'Tuxtla Gutiérrez, Chiapas', 'Janos, Chihuahua', 'Santa María Huatulco, Oaxaca', 'Acapulco de Juárez, Guerrero', 'Hidalgo, Coahuila de Zaragoza', 'San Cristóbal de las Casas, Chiapas', 'Durango, Durango', 'Lázaro Cárdenas, Michoacán', 'Berriozábal, Chiapas']

Total: 11


In [451]:
main_routes = ['pacifico', 'baja', 'sonora', 'chihuahua', 'tejas', 'golfo']

def map_correspondencia(row):
    val = row['correspondencia']
    if val in ['rutas_diferentes', 'ruta_cont_2']:
        return 'Entre rutas paralelas o alternativas'
    elif val == 'ruta_cont_1':
        return 'Sobre una ruta principal'
    elif val == 'o_fuera':
        rutasd = row['rutas_posibles_d']
        if rutasd == 'ninguna':
            return 'Terminar fuera de cualquier ruta'
        rutas_list = [r.strip() for r in rutasd.split(',')]
        if any(r in main_routes for r in rutas_list):
            return 'Sobre una ruta principal'
        else:
            return 'Entre rutas paralelas o alternativas'
    elif val in ['d_fuera', 'ambos_fuera']:
        return 'Terminar fuera de cualquier ruta'

datos_mx['correspondencia_bin'] = datos_mx.apply(map_correspondencia, axis=1)

print(datos_mx['correspondencia_bin'].value_counts())

correspondencia_bin
Sobre una ruta principal                4627
Entre rutas paralelas o alternativas    3661
Terminar fuera de cualquier ruta         541
Name: count, dtype: int64


### Infraestructura de detención

##### Esta tabla que importamos muestra el municipio, entidad, tipo de estancia y año de inicio de operaciones para cada estancia provisional y estación migratoria documentada por la CNDH. Las instalaciones para las cuales la fecha de inicio de operaciones es desconocida son anotadas en 0. 

In [452]:
em = pd.read_csv(REFERENCIAS / 'EMs.csv')

#### Creamos columnas que describen la presencia de infraestructura de detención migratoria en origenes y destinos

In [453]:
# Corregir formatos de CVEGEO
em['CVEGEO'] = em['CVEGEO'].astype(str).str.zfill(5)

# Extraer CVEGEOS de df ems
em_cvegeos = set(em['CVEGEO'].unique())

# Crear em_o columna: 1 si origen_CVEGEO está en ems, 0 si no
datos_mx['em_o'] = datos_mx['origen_cve'].isin(em_cvegeos).astype(int)

# Crear em_d columna: 1 si destino_CVEGEO está en ems, 0 si no
datos_mx['em_d'] = datos_mx['destino_cve'].isin(em_cvegeos).astype(int)

# Checar resultado
columns_to_show = ['lugar_destino', 'lugar_origen', 'em_o', 'em_d']
print(datos_mx[columns_to_show].sample(20))

                      lugar_destino  \
3587             Tenosique, Tabasco   
6616      Tuxtla Gutiérrez, Chiapas   
7288       Pachuca de Soto, Hidalgo   
1079   Iztapalapa, Ciudad de México   
2928             Tapachula, Chiapas   
4439   Iztapalapa, Ciudad de México   
2562             Tapachula, Chiapas   
5128                Centro, Tabasco   
2770   Iztapalapa, Ciudad de México   
749              Acayucan, Veracruz   
8589                Centro, Tabasco   
8537                Centro, Tabasco   
10612               Centro, Tabasco   
8775                Centro, Tabasco   
2547   Iztapalapa, Ciudad de México   
9957                Centro, Tabasco   
704              Acayucan, Veracruz   
5066                Centro, Tabasco   
3443             Acayucan, Veracruz   
954              Tapachula, Chiapas   

                                    lugar_origen  em_o  em_d  
3587   Apetatitlán de Antonio Carvajal, Tlaxcala     0     0  
6616                          Huehuetán, Chiapas     1

In [454]:
# Crear columna em_s para analizar infraestructura de detención migratoria 
def clasificar_ems(row):
    suma = row['em_o'] + row['em_d']
    
    if suma == 0:
        return 'no_em'
    elif suma == 1:
        if row['em_o'] == 1:
            return 'em_o'
        else:
            return 'em_d'
    elif suma == 2:
        return 'ambos'
    else:
        return 'ERROR'  

datos_mx['ems'] = datos_mx.apply(clasificar_ems, axis=1)

# Checar resultado 
print(datos_mx[['em_o', 'em_d', 'ems']].sample(20))

# Y distribución
print("\nDistribución ems:")
print(datos_mx['ems'].value_counts())

      em_o  em_d    ems
7549     1     1  ambos
5698     1     1  ambos
8328     1     0   em_o
6703     1     1  ambos
6888     1     1  ambos
38       0     1   em_d
4607     1     1  ambos
7804     1     1  ambos
1148     1     1  ambos
322      1     1  ambos
1475     1     1  ambos
3434     1     1  ambos
2379     1     1  ambos
9660     0     1   em_d
3074     1     1  ambos
7430     1     1  ambos
5723     1     1  ambos
1561     1     1  ambos
9472     1     1  ambos
278      1     1  ambos

Distribución ems:
ems
ambos    6621
em_d     1496
em_o      581
no_em     131
Name: count, dtype: int64


In [455]:
# Haremos que la columna ems tome en cuenta la fecha de inicio de operaciones cuando sea posible
cvegeo_to_operaciones = dict(zip(em['CVEGEO'], em['operaciones_desde']))

# Recalcular em_o
def check_em_o(row):
    cvegeo = row['origen_cve']
    if cvegeo in cvegeo_to_operaciones:
        if row['year'] >= cvegeo_to_operaciones[cvegeo]:
            return 1
    return 0

# Recalcular em_d
def check_em_d(row):
    cvegeo = row['destino_cve']
    if cvegeo in cvegeo_to_operaciones:
        if row['year'] >= cvegeo_to_operaciones[cvegeo]:
            return 1
    return 0

# Aplicar
datos_mx['em_o'] = datos_mx.apply(check_em_o, axis=1)
datos_mx['em_d'] = datos_mx.apply(check_em_d, axis=1)

# Recalcular ems
datos_mx['ems'] = datos_mx.apply(clasificar_ems, axis=1)

# Y distribución
print("\nDistribución ems:")
print(datos_mx['ems'].value_counts())


Distribución ems:
ems
ambos    5811
em_d     2117
em_o      646
no_em     255
Name: count, dtype: int64


In [456]:
# Checar resultados
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

print(datos_mx[['id_fila_og', 'year', 'origen_limpio_mun', 'destino_limpio_mun', 'em_o', 'em_d', 'ems']].sample(30))

pd.reset_option('display.max_rows')
pd.reset_option('display.max_columns')
pd.reset_option('display.width')
pd.reset_option('display.max_colwidth')

       id_fila_og  year     origen_limpio_mun destino_limpio_mun  em_o  em_d  \
2304         2555  2020              Acayucan          Tapachula     1     1   
149           163  2017          Nuevo Laredo         Cuauhtémoc     1     0   
7015         8246  2023                 Janos         Iztapalapa     1     1   
220           234  2017               Tampico           Acayucan     0     1   
7402         8641  2023        Piedras Negras            Morelia     1     1   
3048         3365  2021              Mazatlán          Tapachula     1     1   
6860         8075  2023               Huixtla          Tapachula     1     1   
7461         8700  2023        Piedras Negras          Guadalupe     1     1   
9217        12355  2023              Acayucan   Tuxtla Gutiérrez     1     1   
6534         7745  2023             Huehuetán   Tuxtla Gutiérrez     1     1   
2936         3231  2021                Puebla          Tapachula     1     1   
10290       13763  2024              Aca

In [457]:
datos_mx.to_csv(DATOS / "procesados" / "datos_analizados_solomx.csv", index=False)